In [1]:
%load_ext autoreload
%autoreload 2

import asyncio

from utils.json_polyfill import install_json
from db.store.cacheservice import *

install_json()

In [2]:
from utils.file import rd
from validate.ragdataset import KGRAGDataset, KGRAGDatasetBlock

dataset: KGRAGDataset = KGRAGDataset.model_validate_json(rd("dataset/data.json"))

current_block: KGRAGDatasetBlock = [i for i in dataset.document_blocks if i.block_id == '5e39a2e5-629c-455e-8fda-c2a147a8641a'][0]
print(len(current_block.questions))
current_files = [Path(i) for i in current_block.document_names]
print(current_block.block_id, current_block.document_names)

110
5e39a2e5-629c-455e-8fda-c2a147a8641a ['dataset/pdf/5e39a2e5-629c-455e-8fda-c2a147a8641a/egrul_extract_2023.pdf', 'dataset/pdf/5e39a2e5-629c-455e-8fda-c2a147a8641a/cyprus_department_of_registrar_returns.pdf', 'dataset/pdf/5e39a2e5-629c-455e-8fda-c2a147a8641a/offshore_incorporations_leak_database.pdf', 'dataset/pdf/5e39a2e5-629c-455e-8fda-c2a147a8641a/trust_declarations_orion_jupiter.pdf', 'dataset/pdf/5e39a2e5-629c-455e-8fda-c2a147a8641a/internal_kyc_control_memo.pdf', 'dataset/pdf/5e39a2e5-629c-455e-8fda-c2a147a8641a/anti_corruption_ngo_pep_report.pdf', 'dataset/pdf/5e39a2e5-629c-455e-8fda-c2a147a8641a/nominee_director_matrix_leak.csv.pdf', 'dataset/pdf/5e39a2e5-629c-455e-8fda-c2a147a8641a/global_sanctions_consolidated_notice.pdf', 'dataset/pdf/5e39a2e5-629c-455e-8fda-c2a147a8641a/corporate_bank_account_opening.pdf', 'dataset/pdf/5e39a2e5-629c-455e-8fda-c2a147a8641a/legal_advisory_memo_structuring.pdf', 'dataset/pdf/5e39a2e5-629c-455e-8fda-c2a147a8641a/draft_divestment_plan_amended

In [7]:
from tasks.search_task import execute_query_task
execute_query_task.delay('Существует ли корпоративная связь между ООО "НефтеГазТранс" и офшором Oceanic Holdings Inc').get()

'## Ответ\n\nДа, между [ООО "НефтеГазТранс"](kg://entity/dd3ef7e39d974567a39dc6f9f664d9ba) и офшорной компанией [Oceanic Holdings Inc](kg://entity/1f722731d3519df931c67f5501631101) существует косвенная корпоративная связь, которая прослеживается через конечного выгодоприобретателя (бенефициара). Связующим звеном выступает физическое лицо — [М.С. Волков](kg://entity/81286721dc543b2c78173477ac6e1c93).\n\nЭта связь подтверждается следующим образом:\n1. Зарегистрированная в Панаме компания [Oceanic Holdings Inc](kg://entity/1f722731d3519df931c67f5501631101) принадлежит (факт владения) двум трастам: [Orion Family Trust](kg://entity/2638c65174e4f8c19ef8da239283fe17) [(см. факт владения)](kg://fact/334d6a1df409d5f5a19171c16420a1e4) и [Jupiter Asset Trust](kg://entity/a4325af7febe62fce0bfc637b5f4b8e3) [(см. факт владения)](kg://fact/1e9bf5b72231781cc793e3643c6e3d59).\n2. Выгодоприобретателем (бенефициаром) обоих указанных трастов является вышеупомянутый [М.С. Волков](kg://entity/81286721dc543b

In [ ]:
# from load.top import ingest_docs
# ingest_docs(current_files)

In [20]:
from validate.demoimpl.vectorrag import VectorRAG
rn = VectorRAG()
await rn.load(current_files)

/home/petr/study/diploma/.venv/lib/python3.12/site-packages/milvus_lite/__init__.py:15: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


In [22]:
from validate.demoimpl.graphrag import OtherGraphRAG
rl = OtherGraphRAG()
await rl.load(current_files)

INFO: [] Loaded graph from ./light_rag/04b855858c55331b1721138f13cd7732/graph_chunk_entity_relation.graphml with 759 nodes, 1251 edges
INFO:nano-vectordb:Load (759, 1536) data
INFO:nano-vectordb:Init {'embedding_dim': 1536, 'metric': 'cosine', 'storage_file': './light_rag/04b855858c55331b1721138f13cd7732/vdb_entities.json'} 759 data
INFO:nano-vectordb:Load (1251, 1536) data
INFO:nano-vectordb:Init {'embedding_dim': 1536, 'metric': 'cosine', 'storage_file': './light_rag/04b855858c55331b1721138f13cd7732/vdb_relationships.json'} 1251 data
INFO:nano-vectordb:Load (51, 1536) data
INFO:nano-vectordb:Init {'embedding_dim': 1536, 'metric': 'cosine', 'storage_file': './light_rag/04b855858c55331b1721138f13cd7732/vdb_chunks.json'} 51 data
INFO: [] Process 15765 KV load full_docs with 11 records
INFO: [] Process 15765 KV load text_chunks with 51 records
INFO: [] Process 15765 KV load full_entities with 11 records
INFO: [] Process 15765 KV load full_relations with 11 records
INFO: [] Process 15765 

I'm sorry, I don't understand your request. Could you please provide more details or ask a specific question? I can provide information about OOO "TechPromAlliance" based on the provided context if you'd like.


In [5]:
from celery import group
from tasks.search_task import execute_query_task

async def _ask(m, q):
    return await asyncio.gather(*[m.ask(i.question) for i in q])

rn_answers = await _ask(rn, current_block.questions)
rl_answers = await _ask(rl, current_block.questions)
rm_answers = group([execute_query_task.s(q.question) for q in current_block.questions])().get()

KeyboardInterrupt: 

In [ ]:
# 48m36s

In [70]:
ans_labels = [
    "naive",
    "light",
    "mine"
]
ans_data = [
    rn_answers,
    rl_answers,
    rm_answers
]

In [74]:
from validate.metrics.val_compare import evaluate_comparing_out_mrr
from validate.metrics.val_trick import evaluate_tricky
from validate.metrics.val_precision import evaluate_precision
from validate.metrics.val_recall import evaluate_recall
import pandas as pd

async def evaluate(block: KGRAGDatasetBlock, answers: List[str], full=False):
    to_check = [(desc, ans) for ans, desc in zip(answers, block.questions)]
    q0 = [(d, r) for d, r in to_check if not d.is_trick]
    recall = asyncio.gather(*[
        evaluate_recall(d, r)
        for d, r in q0
    ])
    precision = asyncio.gather(*[
        evaluate_precision(block, d, r)
        for d, r in q0
    ])
    tricky = evaluate_tricky(block.questions, answers)

    def _et(x):
        x = x.type
        return dict(subgraph="bridge", fan_in="bridge").get(x, x)

    r, p, t = await asyncio.gather(recall, precision, tricky)
    rp = pd.DataFrame.from_dict([{**x0, **x1, "type": _et(qi[0])} for x0, x1, qi in zip(r, p, q0)])
    out = rp.describe().transpose()[["mean", "std"]]
    out.loc['tricky_precision'] = t
    return rp if full else out

async def evaluate_all(block: KGRAGDatasetBlock, all_ans):
    ft = [evaluate(block, i, False) for i in all_ans] + [evaluate_comparing_out_mrr(current_block.questions, all_ans)]
    *res, comp = await asyncio.gather(*ft)
    final = pd.concat([i[["mean"]].rename(columns=dict(mean=l)) for i, l in zip(res, ans_labels)], axis=1)
    final.loc['compare_mrr'] = comp
    return final

In [75]:
final = await evaluate_all(current_block, ans_data)
final

/home/petr/study/diploma/.venv/lib/python3.12/site-packages/langchain_community/cache.py:272: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  return [loads(row[0]) for row in rows]
/home/petr/study/diploma/.venv/lib/python3.12/site-packages/langchain_community/cache.py:272: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  return [loads(row[0]) for row in rows]
/home/petr/study/diploma/.venv/lib/python3.12/site-packages/langchain_community/cache.py:272: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes

,naive,light,mine
recall_entity,0.472207,0.814698,0.943864
recall_fact_extract,0.248230,0.636813,0.676038
recall_fact_derive,0.403846,0.750000,0.804487
recall,0.305209,0.685125,0.816217
precision,0.689535,0.914316,0.921123
hallucination,0.310465,0.085684,0.078877
tricky_precision,0.812500,0.875000,1.000000
compare_mrr,0.405556,0.566667,0.861111


In [65]:
res_full = await asyncio.gather(*[evaluate(current_block, i, True) for i in ans_data])

/home/petr/study/diploma/.venv/lib/python3.12/site-packages/langchain_community/cache.py:272: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  return [loads(row[0]) for row in rows]
/home/petr/study/diploma/.venv/lib/python3.12/site-packages/langchain_community/cache.py:272: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  return [loads(row[0]) for row in rows]
/home/petr/study/diploma/.venv/lib/python3.12/site-packages/langchain_community/cache.py:272: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes

In [66]:
def _ppd(r):
    r = r.groupby("type").mean()
    r["precision"] = 1 - r["hallucination"]
    return r[["recall", "recall_entity", "precision"]]

_ppd(res_full[0]) # light

,recall,recall_entity,precision
type,,,
bridge,0.711320,0.859127,0.991453
n_hop,0.671257,0.791176,0.873478


In [67]:
_ppd(res_full[1]) # mine

,recall,recall_entity,precision
type,,,
bridge,0.752405,0.910053,0.993464
n_hop,0.850000,0.961765,0.882825
